# P96 — Feature Engineering Pipeline v9
**Script:** `enhance_master_table_v9.ipynb`  
**Input:** `master_training_table_v2_with_aadt.csv`  
**Output:** `master_training_table_v9.csv`

### What changed v8 → v9

| | Column | Change | Reason |
|---|---|---|---|
| ➕ | `LIGHT_CONDITION` | Added raw | Replaces `darkness_score` — range 33.6% to **51.3%** severe |
| ➕ | `SPEED_ZONE` | Added raw | Was dropped in v8 despite being in v2 source |
| ➕ | `has_pedestrian` | Added | Was in v2 source but missing from v8 |
| ➕ | `has_cyclist` | Added | Was in v2 source but missing from v8 |
| ➕ | `is_foggy` | Added | Kept for fog interaction traceability |
| ➕ | `fog_night_cold_rural` | New engineered | Fog only dangerous: rural + night + cold (47.6% severe) |
| ➕ | `fog_night_cold_urban` | New engineered | Fog mildly dangerous: urban + night + cold (40.0% severe) |
| ➖ | `darkness_score` | Removed | Manual 0–1 map of LIGHT_CONDITION — got ordering wrong |

> **Run cells top to bottom.**  
> Input file must be at `../output/master_training_table_v2_with_aadt.csv` relative to this notebook.

## Step 0 — Imports & setup

In [4]:
import sys, io
import os
import numpy as np
import pandas as pd

print("Libraries imported.")

Libraries imported.


## Step 1 — Paths

Expected folder structure:
```
heng/
├── output/
│   ├── master_training_table_v2_with_aadt.csv   ← input (from merge_all.py)
│   └── master_training_table_v9.csv             ← output (this notebook)
└── src/
    └── enhance_master_table_v9.ipynb            ← this file
```

In [5]:
# ── Adjust if running from a different directory ──
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
IN_PATH   = os.path.join(REPO_ROOT, 'output', 'master_training_table_v2_with_aadt.csv')
OUT_PATH  = os.path.join(REPO_ROOT, 'output', 'master_training_table_v9.csv')

print(f"Input : {os.path.abspath(IN_PATH)}")
print(f"Output: {os.path.abspath(OUT_PATH)}")

Input : /Users/ronuts/Desktop/2026-Sem1/COS40005-Computing Technology Project A/trainnn-main/heng/output/master_training_table_v2_with_aadt.csv
Output: /Users/ronuts/Desktop/2026-Sem1/COS40005-Computing Technology Project A/trainnn-main/heng/output/master_training_table_v9.csv


## Step 2 — Load data
Read `master_training_table_v2_with_aadt.csv` — the raw merged dataset from `merge_all.py`.  
This is the source for all features. It has 35 columns; v9 selects and engineers from these.

In [6]:
df = pd.read_csv(IN_PATH, low_memory=False)
print(f"Loaded: {len(df):,} rows x {len(df.columns)} columns")
print(f"\nSource columns ({len(df.columns)}):")
for i, c in enumerate(df.columns):
    print(f"  {i+1:2d}. {c}")

Loaded: 94,398 rows x 35 columns

Source columns (35):
   1. crash_hour
   2. DAY_OF_WEEK
   3. is_weekend
   4. is_public_holiday
   5. SPEED_ZONE
   6. LIGHT_CONDITION
   7. ROAD_GEOMETRY_DESC
   8. ROAD_NAME
   9. ROAD_TYPE
  10. ROAD_NAME_INT
  11. DISTANCE_LOCATION
  12. LATITUDE
  13. LONGITUDE
  14. LGA_NAME
  15. DEG_URBAN_NAME
  16. POSTCODE_CRASH
  17. nearest_city
  18. total_persons
  19. has_pedestrian
  20. has_cyclist
  21. temperature_2m
  22. precipitation
  23. wind_speed_10m
  24. wind_gusts_10m
  25. cloud_cover
  26. is_foggy
  27. nearest_school_dist_m
  28. in_school_zone
  29. school_zone_active
  30. severe_crash
  31. NO_OF_VEHICLES
  32. NODE_TYPE
  33. road_class
  34. wet_road
  35. aadt_volume


## Step 3 — Fix severe_crash & sample weights

- `severe_crash` → cast to int (0 or 1)
- `sample_weight` → **1.5** for severe, **1.0** for non-severe

This tells the model to pay 50% more attention to severe crash rows.  
Already present in v8 — kept identical here.

In [7]:
df['severe_crash']  = df['severe_crash'].astype(int)
df['sample_weight'] = df['severe_crash'].map({1: 1.5, 0: 1.0})

print(f"severe=1: {(df['severe_crash']==1).sum():,} rows  weight=1.5")
print(f"severe=0: {(df['severe_crash']==0).sum():,} rows  weight=1.0")

severe=1: 34,582 rows  weight=1.5
severe=0: 59,816 rows  weight=1.0


## Step 4 — Data quality fixes

Same fixes as v8 — unchanged.

| Column | Problem | Fix |
|---|---|---|
| `SPEED_ZONE` | 777 / 888 = sentinel "unknown" codes | → 60 (median urban speed) |
| `DAY_OF_WEEK` | 0 is invalid (valid range 1–7) | → 4 (Wednesday midweek default) |
| `NODE_TYPE` | 'U' = undefined | → 'N' (non-intersection) |
| `NO_OF_VEHICLES` | 0 vehicles impossible | → 1 |

In [8]:
if 'SPEED_ZONE' in df.columns:
    bad = df['SPEED_ZONE'].isin([777, 888]).sum()
    df.loc[df['SPEED_ZONE'].isin([777, 888]), 'SPEED_ZONE'] = 60
    print(f"SPEED_ZONE    : fixed {bad} sentinel values (777/888 → 60)")

if 'DAY_OF_WEEK' in df.columns:
    bad = (df['DAY_OF_WEEK'] == 0).sum()
    df.loc[df['DAY_OF_WEEK'] == 0, 'DAY_OF_WEEK'] = 4
    print(f"DAY_OF_WEEK   : fixed {bad} invalid zeros → 4")

if 'NODE_TYPE' in df.columns:
    bad = (df['NODE_TYPE'] == 'U').sum()
    df.loc[df['NODE_TYPE'] == 'U', 'NODE_TYPE'] = 'N'
    print(f"NODE_TYPE     : merged {bad} 'U' → 'N'")

if 'NO_OF_VEHICLES' in df.columns:
    bad = (df['NO_OF_VEHICLES'] == 0).sum()
    df.loc[df['NO_OF_VEHICLES'] == 0, 'NO_OF_VEHICLES'] = 1
    print(f"NO_OF_VEHICLES: fixed {bad} zeros → 1")

SPEED_ZONE    : fixed 942 sentinel values (777/888 → 60)
DAY_OF_WEEK   : fixed 3291 invalid zeros → 4
NODE_TYPE     : merged 1 'U' → 'N'
NO_OF_VEHICLES: fixed 6 zeros → 1


## Step 5 — Feature engineering (time & speed)

Same as v8 — unchanged.

| Feature | How | Why |
|---|---|---|
| `speed_risk` | SPEED_ZONE mapped 0–6 | Ordinal risk: 0=no zone → 6=110km/h |
| `hour_sin/cos` | sin/cos of crash_hour/24 | Treats hour as cycle — 23h and 0h are neighbours |
| `day_sin/cos` | sin/cos of DAY_OF_WEEK/7 | Treats weekday as cycle |
| `is_peak_hour` | 1 if hour in 7-9am or 4-6pm | Morning/afternoon peak traffic |

> **`darkness_score` has been removed.** `LIGHT_CONDITION` raw replaces it — see Step 6.

In [9]:
SPEED_RISK_MAP = {0:0, 40:1, 50:2, 60:3, 75:4, 80:4, 90:5, 100:6, 110:6}
df['speed_risk']   = df['SPEED_ZONE'].map(SPEED_RISK_MAP).fillna(3).astype(int)
df['hour_sin']     = np.sin(2 * np.pi * df['crash_hour'] / 24)
df['hour_cos']     = np.cos(2 * np.pi * df['crash_hour'] / 24)
df['is_peak_hour'] = df['crash_hour'].isin([7,8,9,16,17,18]).astype(int)
df['day_sin']      = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['day_cos']      = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)

print("Engineered: speed_risk, hour_sin/cos, day_sin/cos, is_peak_hour")
print(f"\nspeed_risk distribution:")
print(df['speed_risk'].value_counts().sort_index().to_string())

Engineered: speed_risk, hour_sin/cos, day_sin/cos, is_peak_hour

speed_risk distribution:
speed_risk
0    12498
1     7020
2    15155
3    30984
4    15258
5      125
6    13358


## Step 6 — LIGHT_CONDITION replaces darkness_score (v9 change)

`darkness_score` in v8 was a hand-crafted 0–1 mapping of `LIGHT_CONDITION`:
```python
DARKNESS_MAP = {1:0.00, 2:0.30, 3:0.60, 4:0.85, 5:1.00, 6:0.70, 9:0.40}
```

**The problem:** code 4 was mapped *higher* (0.85) than code 3 (0.60),  
but code 3 is actually **more dangerous** in the real data (40.3% vs 38.7% severe).  
The model was being given incorrect ordering information.

**The fix:** keep `LIGHT_CONDITION` raw — let the model learn the real pattern itself.

| Code | Meaning | Severe rate | darkness_score (old) |
|---|---|---|---|
| 1 | Daylight | 36.2% | 0.00 |
| 2 | Dusk / dawn | 33.6% | 0.30 |
| 3 | Dark — street lights **on** | 40.3% | 0.60 |
| 4 | Dark — no street lights | 38.7% | 0.85 ← **wrong — higher than code 3** |
| 5 | Dark — street lights **off** | **51.3%** | 1.00 |
| 6 | Dark — unknown | 26.6% | 0.70 |
| 9 | Unknown | 14.3% | 0.40 |

Code 5 (lights physically switched off) is the most dangerous — 51.3% severe.  
This is 37 percentage points above code 9. No other single feature has this range.

In [10]:
# Verify LIGHT_CONDITION rates in this dataset
lc = df.groupby('LIGHT_CONDITION')['severe_crash'].agg(['mean','count'])
lc.columns = ['severe_rate_%', 'count']
lc['severe_rate_%'] = (lc['severe_rate_%'] * 100).round(1)
lc_map = {1:'Daylight', 2:'Dusk/dawn', 3:'Dark-lights on',
          4:'Dark-no lights', 5:'Dark-lights OFF', 6:'Dark-unknown', 9:'Unknown'}
lc['meaning'] = lc.index.map(lc_map)
print("LIGHT_CONDITION — confirming code 5 is most dangerous:")
print(lc[['meaning', 'count', 'severe_rate_%']].to_string())

LIGHT_CONDITION — confirming code 5 is most dangerous:
                         meaning  count  severe_rate_%
LIGHT_CONDITION                                       
1                       Daylight  64456           36.2
2                      Dusk/dawn   6624           33.6
3                 Dark-lights on  14290           40.3
4                 Dark-no lights    199           38.7
5                Dark-lights OFF   4744           51.3
6                   Dark-unknown   1159           26.6
9                        Unknown   2926           14.3


## Step 7 — Fog interaction features (v9 new)

### Why is_foggy alone has no signal
| Condition | Severe rate |
|---|---|
| `is_foggy = 0` | 36.7% |
| `is_foggy = 1` | 36.1% |

Fog alone is actually slightly *safer* — drivers slow down when they see it.

### When fog becomes dangerous
Fog only matters when combined with **night + cold temperature (winter)**.  
And even then, **location matters** — your observation was correct:

| Scenario | Severe rate |
|---|---|
| Baseline | 36.6% |
| Fog + night + cold — **urban** | 40.0% |
| Fog + night + cold — **rural** | **47.6%** |

**Why urban fog is less dangerous:**  
Dense housing and buildings around urban intersections provide ambient light  
from houses, shops, and street corners — fog's visibility effect is suppressed.

**Why rural fog is more dangerous:**  
No infrastructure, no ambient light, higher speeds (80–100km/h),  
and drivers traveling long distances add fatigue on top of the fog.

So we engineer **two** separate columns instead of one `fog_night_cold`.

In [11]:
# Temporary helper columns (prefixed _ so they're easy to drop)
df['_is_night'] = (df['crash_hour'].between(20, 23) | df['crash_hour'].between(0, 5)).astype(int)
df['_is_cold']  = (df['temperature_2m'] < 10).astype(int)
df['_is_rural'] = (df['DEG_URBAN_NAME'] == 'RURAL_VICTORIA').astype(int)

# fog_night_cold_rural: strongest fog signal — rural roads at night in winter
df['fog_night_cold_rural'] = (
    (df['is_foggy'] == 1) &
    (df['_is_night'] == 1) &
    (df['_is_cold']  == 1) &
    (df['_is_rural'] == 1)
).astype(int)

# fog_night_cold_urban: weaker but still above baseline
df['fog_night_cold_urban'] = (
    (df['is_foggy'] == 1) &
    (df['_is_night'] == 1) &
    (df['_is_cold']  == 1) &
    (df['_is_rural'] == 0)
).astype(int)

# Drop temp columns — only the interactions go into the final dataset
df = df.drop(columns=['_is_night', '_is_cold', '_is_rural'])

# Verify
for col in ['fog_night_cold_rural', 'fog_night_cold_urban']:
    n    = df[col].sum()
    rate = df[df[col] == 1]['severe_crash'].mean() * 100
    print(f"{col:<30}: {n:>4,} rows  severe={rate:.1f}%  (baseline=36.6%)")

fog_night_cold_rural          :  170 rows  severe=47.6%  (baseline=36.6%)
fog_night_cold_urban          :  488 rows  severe=40.0%  (baseline=36.6%)


## Step 8 — Calendar columns (is_school_holiday, is_daylight_saving)

`is_school_holiday` and `is_daylight_saving` were engineered by **Hareet's calendar pipeline**  
and exist in `master_training_table_v8.csv` but NOT in the v2 source file.

If you are running this notebook from the v2 source directly, these will be missing  
and filled with 0. To get real values, merge Hareet's calendar output before running.

In [12]:
HAREET_COLS = ['is_school_holiday', 'is_daylight_saving']
for col in HAREET_COLS:
    if col not in df.columns:
        df[col] = 0
        print(f"NOTE: '{col}' not in source — filled with 0.")
        print(f"      Merge Hareet's calendar pipeline output to fix this.")
    else:
        print(f"OK  : '{col}' found in source — using real values.")

NOTE: 'is_school_holiday' not in source — filled with 0.
      Merge Hareet's calendar pipeline output to fix this.
NOTE: 'is_daylight_saving' not in source — filled with 0.
      Merge Hareet's calendar pipeline output to fix this.


## Step 9 — Crash rate

Same as v8 — unchanged.

Formula: `crashes on road / (aadt_volume × 365 × 7) × 1,000,000`  
= crashes per million vehicle passes on that road segment.

In [13]:
if 'aadt_volume' in df.columns:
    df['_road_key'] = df['aadt_volume'].astype(str) + '_' + df['road_class'].astype(str)
    cpk = df.groupby('_road_key').size().reset_index()
    cpk.columns = ['_road_key', '_crashes_on_road']
    df = df.merge(cpk, on='_road_key', how='left')
    total_passes = df['aadt_volume'] * 365 * 7
    df['crash_rate'] = (
        df['_crashes_on_road'] / total_passes.replace(0, np.nan) * 1_000_000
    ).fillna(0)
    df = df.drop(columns=['_road_key', '_crashes_on_road'])
    print(f"crash_rate range : {df['crash_rate'].min():.2f} to {df['crash_rate'].max():.2f}")
    print(f"crash_rate zeros : {(df['crash_rate']==0).sum():,}")
else:
    df['crash_rate'] = 0.0
    print("WARNING: aadt_volume missing — crash_rate set to 0")

crash_rate range : 0.00 to 260.93
crash_rate zeros : 31,974


## Step 10 — Select final columns

### v8 → v9 column changes

| Column | v8 | v9 | Note |
|---|---|---|---|
| `darkness_score` | ✅ | ❌ | Removed — lossy approximation of LIGHT_CONDITION |
| `LIGHT_CONDITION` | ❌ | ✅ | Raw code 1–9, true severe rate range 33.6–51.3% |
| `SPEED_ZONE` | ❌ | ✅ | Was in v2 source but dropped in v8 — restored |
| `has_pedestrian` | ❌ | ✅ | Was in v2 source but dropped in v8 — restored |
| `has_cyclist` | ❌ | ✅ | Was in v2 source but dropped in v8 — restored |
| `is_foggy` | ❌ | ✅ | Added for fog interaction traceability |
| `fog_night_cold_rural` | ❌ | ✅ | New — rural fog at night/cold: 47.6% severe |
| `fog_night_cold_urban` | ❌ | ✅ | New — urban fog at night/cold: 40.0% severe |
| All others | ✅ | ✅ | Unchanged from v8 |

In [14]:
# CHANGES vs v8:
#   + LIGHT_CONDITION       raw — replaces darkness_score (wider 33.6%–51.3% range)
#   + SPEED_ZONE            restored — was in v2 but missing from v8
#   + has_pedestrian        restored — was in v2 but missing from v8
#   + has_cyclist           restored — was in v2 but missing from v8
#   + is_foggy              added — traceability for fog interactions
#   + fog_night_cold_rural  new — rural fog at night/cold (47.6% severe)
#   + fog_night_cold_urban  new — urban fog at night/cold (40.0% severe)
#   - darkness_score        removed — manual 0-1 map with wrong ordering

FINAL_COLUMNS = [
    'severe_crash', 'sample_weight',

    # Time features (same as v8)
    'is_weekend', 'is_peak_hour', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'is_public_holiday', 'is_school_holiday', 'is_daylight_saving',

    # Road features — SPEED_ZONE restored (was in v2, missing from v8)
    'SPEED_ZONE', 'speed_risk',
    'ROAD_GEOMETRY_DESC', 'DISTANCE_LOCATION', 'NODE_TYPE', 'road_class',
    'aadt_volume', 'crash_rate',

    # Lighting — LIGHT_CONDITION raw replaces darkness_score
    # 1=Daylight(36.2%)  2=Dusk/dawn(33.6%)  3=Dark lights on(40.3%)
    # 4=Dark no lights(38.7%)  5=Dark lights OFF(51.3%)  6=Dark unknown(26.6%)
    'LIGHT_CONDITION',

    # Weather
    'wet_road',              # rain — road surface wet
    'is_foggy',              # fog flag — kept for interaction traceability
    'fog_night_cold_rural',  # fog only dangerous: rural + night (20-5h) + cold (<10°C)
    'fog_night_cold_urban',  # fog mildly dangerous: urban + night + cold

    # Location & crash context
    'LGA_NAME', 'DEG_URBAN_NAME',
    'NO_OF_VEHICLES',
    'nearest_school_dist_m',
    'has_pedestrian', 'has_cyclist',  # restored from v2 — missing in v8
]

# Check for any missing columns
missing = [c for c in FINAL_COLUMNS if c not in df.columns]
if missing:
    print(f"WARNING: still missing — filled with 0: {missing}")
    for c in missing:
        df[c] = 0
else:
    print("All columns present — no warnings.")

df_out = df[FINAL_COLUMNS].copy()
print(f"\nv9 shape: {df_out.shape}  ({len(FINAL_COLUMNS)} columns)")

All columns present — no warnings.

v9 shape: (94398, 30)  (30 columns)


## Step 11 — Fill nulls
Numeric nulls → column median. String nulls → 'Unknown'.  
Assert zero nulls remain before saving.

In [15]:
for col in df_out.select_dtypes(include=[np.number]).columns:
    n_null = df_out[col].isnull().sum()
    if n_null > 0:
        df_out[col] = df_out[col].fillna(df_out[col].median())
        print(f"  {col}: filled {n_null} nulls with median")

for col in df_out.select_dtypes(include=['object']).columns:
    n_null = df_out[col].isnull().sum()
    if n_null > 0:
        df_out[col] = df_out[col].fillna('Unknown')
        print(f"  {col}: filled {n_null} nulls with 'Unknown'")

assert df_out.isnull().sum().sum() == 0, "Nulls remain — check pipeline above"
print("Nulls: 0 ✓")

Nulls: 0 ✓


/var/folders/yv/zft6fqb95_lbgz_sk8qz1dlm0000gp/T/ipykernel_1563/2115974697.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_out.select_dtypes(include=['object']).columns:


## Step 12 — Save `master_training_table_v9.csv`

In [16]:
df_out.to_csv(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")

Saved → /Users/ronuts/Desktop/2026-Sem1/COS40005-Computing Technology Project A/trainnn-main/heng/output/master_training_table_v9.csv


## Done — Summary

In [17]:
print("=" * 58)
print("  ENHANCE_MASTER_TABLE v9 COMPLETE")
print("=" * 58)
print(f"  Input          : master_training_table_v2_with_aadt.csv")
print(f"  Output         : master_training_table_v9.csv")
print(f"  Rows           : {len(df_out):,}")
print(f"  Columns        : {len(df_out.columns)}  (v8 had 24)")
print(f"  Nulls          : {df_out.isnull().sum().sum()}")
print(f"  severe=1       : {(df_out['severe_crash']==1).sum():,}  ({(df_out['severe_crash']==1).mean()*100:.1f}%)")
print(f"  severe=0       : {(df_out['severe_crash']==0).sum():,}  ({(df_out['severe_crash']==0).mean()*100:.1f}%)")
print(f"  aadt zeros     : {(df_out['aadt_volume']==0).sum():,}")
print(f"  crash_rate zero: {(df_out['crash_rate']==0).sum():,}")
print()
print(f"  Columns ({len(df_out.columns)}):")
for i, col in enumerate(df_out.columns):
    tag = ' ← NEW' if col in ['LIGHT_CONDITION','SPEED_ZONE','has_pedestrian',
                               'has_cyclist','is_foggy',
                               'fog_night_cold_rural','fog_night_cold_urban'] else ''
    tag = ' ← REMOVED' if col == 'darkness_score' else tag
    print(f"    {i+1:2d}. {col}{tag}")
print()
print("  Next step: open train_model.ipynb")
print("  Change DATA_PATH to master_training_table_v9.csv and rerun.")

  ENHANCE_MASTER_TABLE v9 COMPLETE
  Input          : master_training_table_v2_with_aadt.csv
  Output         : master_training_table_v9.csv
  Rows           : 94,398
  Columns        : 30  (v8 had 24)
  Nulls          : 0
  severe=1       : 34,582  (36.6%)
  severe=0       : 59,816  (63.4%)
  aadt zeros     : 31,974
  crash_rate zero: 31,974

  Columns (30):
     1. severe_crash
     2. sample_weight
     3. is_weekend
     4. is_peak_hour
     5. hour_sin
     6. hour_cos
     7. day_sin
     8. day_cos
     9. is_public_holiday
    10. is_school_holiday
    11. is_daylight_saving
    12. SPEED_ZONE ← NEW
    13. speed_risk
    14. ROAD_GEOMETRY_DESC
    15. DISTANCE_LOCATION
    16. NODE_TYPE
    17. road_class
    18. aadt_volume
    19. crash_rate
    20. LIGHT_CONDITION ← NEW
    21. wet_road
    22. is_foggy ← NEW
    23. fog_night_cold_rural ← NEW
    24. fog_night_cold_urban ← NEW
    25. LGA_NAME
    26. DEG_URBAN_NAME
    27. NO_OF_VEHICLES
    28. nearest_school_dist_m
    